# ▶ Full reproduction — one notebook

Reproduces the **entire** project end-to-end on Yambda-50M: data → baselines →
SASRec → joint-fusion hybrid (negative result) → late fusion → robustness → tail
analysis, ending in one summary table. SASRec is trained **once** and reused, so
this is faster than running the per-stage notebooks.

**How to run:** *Settings → Accelerator =* **GPU** (T4), *Internet =* **On**, then
**Run All**. Total ≈ 45–55 min on a T4. (Optional: `HF_TOKEN` in *Add-ons → Secrets*.)

Code: https://github.com/ak1232320/nndl_capstone_2026

In [ ]:
!pip install -q --no-cache-dir --upgrade "ymrec[baselines] @ git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
import os
os.environ["YMREC_DATA"] = "/kaggle/working/data"  # 13.8 GB embeddings -> 20 GB working disk
import time, numpy as np, pandas as pd, torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
from ymrec.config import TOPK
K = max(TOPK)
EPOCHS = 120          # SASRec / hybrid epochs (lower for a quick smoke run)
results = {}          # model -> NDCG@10, filled as we go
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 1 · Data — Global Temporal Split, Listen+, shared vocab

`prep` builds the sparse user×item matrix (baselines); `build_sequences` builds
per-user histories (SASRec); both share the same split + item vocabulary. Audio
embeddings are filtered to that vocab.

In [ ]:
from ymrec.data.prep import prepare
from ymrec.data.sequences import build_sequences
from ymrec.data.embeddings import load_aligned_embeddings

p = prepare(size="50m")
data = build_sequences(size="50m", maxlen=200)
content_emb, content_mask, dim = load_aligned_embeddings(data.item_ids)
assert np.array_equal(p.item_ids, data.item_ids)  # baselines & neural models share one vocab
print(f"users={p.n_users:,}  items={p.n_items:,}  eval_users={len(p.eval_user_idx):,}  "
      f"audio_dim={dim}  audio_coverage={content_mask.mean():.3f}")

## 2 · Baselines — MostPop & ItemKNN

In [ ]:
from ymrec.eval.metrics import evaluate_ranking
from ymrec.baselines.item_knn import fit_recommend

def ndcg(recs):
    return evaluate_ranking(recs, p.relevant, n_items=p.n_items, ks=TOPK)["ndcg@10"]

pop = np.asarray(p.train_ui.sum(axis=0)).ravel()
mostpop = np.tile(p.item_ids[np.argsort(-pop)[:K]], (len(p.eval_user_idx), 1))
results["MostPop"] = ndcg(mostpop)
print(f"MostPop          NDCG@10 = {results['MostPop']:.4f}")

for v in ["cosine", "tfidf", "bm25"]:
    t = time.time()
    recs = fit_recommend(p.train_ui, p.eval_user_idx, p.item_ids, K, neighbors=100, variant=v)
    results[f"ItemKNN-{v}"] = ndcg(recs)
    print(f"ItemKNN-{v:<6}   NDCG@10 = {results[f'ItemKNN-{v}']:.4f}   [{time.time()-t:.0f}s]")

## 3 · SASRec (sequence model) — trained once, reused below

In [ ]:
from ymrec.models.sasrec import train_and_eval as train_sasrec
t = time.time()
sasrec, sasrec_best = train_sasrec(data, d=64, n_blocks=2, n_heads=1, dropout=0.2,
                                   epochs=EPOCHS, batch_size=128, lr=1e-3, eval_every=20)
results["SASRec"] = sasrec_best["ndcg@10"]
print(f"\nSASRec NDCG@10 = {results['SASRec']:.4f}  [{(time.time()-t)/60:.1f} min]")

## 4 · Hybrid, attempt 1 — joint embedding fusion (negative result)

Content folded into the item embedding and trained end-to-end. Expected to
**overfit and lose** to SASRec — motivating the late-fusion design next.

In [ ]:
from ymrec.models.hybrid import train_and_eval as train_hybrid
t = time.time()
_, hyb_best = train_hybrid(data, content_emb, content_mask, d=64, n_blocks=2, n_heads=1,
                           dropout=0.2, epochs=EPOCHS, batch_size=128, lr=1e-3, eval_every=20)
results["Hybrid (joint fusion)"] = hyb_best["ndcg@10"]
print(f"\nHybrid joint NDCG@10 = {results['Hybrid (joint fusion)']:.4f}  "
      f"(vs SASRec {results['SASRec']:.4f})  [{(time.time()-t)/60:.1f} min]")

## 5 · Hybrid, attempt 2 — late fusion (the winner)

Frozen SASRec + a frozen content score, weight β tuned on a **validation** split
of users, reported on held-out test users. β = 0 recovers SASRec, so it cannot lose.

In [ ]:
from ymrec.models.fusion import tune_fusion
fus = tune_fusion(sasrec, content_emb, data, val_frac=0.5, seed=0)
results["Hybrid (late fusion)"] = fus["test_fused"]["ndcg@10"]
print(f"\nbest beta = {fus['best_beta']}")
print(f"test split: SASRec-only {fus['test_sasrec_only']['ndcg@10']:.4f}  ->  "
      f"fused {fus['test_fused']['ndcg@10']:.4f}")

## 6 · Robustness — 5 held-out user splits

In [ ]:
from ymrec.models.fusion import robustness
rob = robustness(sasrec, content_emb, data, seeds=(0, 1, 2, 3, 4))
s = rob["summary"]
print(f"\nSASRec {s['sasrec_mean']:.4f}±{s['sasrec_std']:.4f}  ->  "
      f"Fused {s['fused_mean']:.4f}±{s['fused_std']:.4f}   "
      f"lift {s['lift_mean_%']:+.1f}% ± {s['lift_std_%']:.1f}%")

## 7 · Tail analysis — where does the audio content help?

In [ ]:
from ymrec.models.fusion import tail_analysis
tail = tail_analysis(sasrec, content_emb, data, beta=fus["best_beta"], thresholds=(5, 20, 100))
pd.DataFrame(tail)

## 8 · Summary — full model ladder (NDCG@10)

In [ ]:
ladder = pd.Series(results, name="NDCG@10").sort_values().round(4)
print(ladder.to_string())
print(f"\nHeadline: late-fusion hybrid {s['fused_mean']:.4f} vs SASRec {s['sasrec_mean']:.4f} "
      f"= {s['lift_mean_%']:+.1f}% ± {s['lift_std_%']:.1f}% (robust across 5 splits).")
ladder.to_frame()